# 10. CatBoost
### Gradient Boosting with Categorical Features

**CatBoost** (developed by Yandex) is specifically designed to handle **categorical features** seamlessly without manual one-hot encoding or target leakage.

#### Key Innovations:
1. **Ordered Target Encoding**: Uses online target statistics to encode categorical features on the fly without causing target leakage.
2. **Oblivious (Symmetric) Decision Trees**: Uses the same splitting criterion across an entire tree level, providing balanced trees and fast GPU inference.
3. **Ordered Boosting**: Solves prediction shift caused by standard boosting algorithms.


In [ ]:
# Step 1: Install CatBoost if needed & Import Libraries
# !pip install catboost

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

try:
    from catboost import CatBoostClassifier, Pool
    print("CatBoost successfully imported.")
except ImportError:
    print("CatBoost is not installed. Please run: pip install catboost")


In [ ]:
# Step 2: Create a Dataset with both Numerical and Categorical Features
# Example: Customer churn prediction scenario
np.random.seed(42)
n_samples = 600

data = pd.DataFrame({
    'age': np.random.randint(18, 70, n_samples),
    'salary': np.random.normal(50000, 15000, n_samples).round(2),
    'credit_score': np.random.randint(300, 850, n_samples),
    'contract_type': np.random.choice(['Month-to-Month', 'One-Year', 'Two-Year'], n_samples),
    'payment_method': np.random.choice(['Credit Card', 'Bank Transfer', 'Electronic Check'], n_samples),
})

# Target: Probability of churn influenced by contract and age
logit = -2.0 + (data['contract_type'] == 'Month-to-Month') * 1.5 - (data['salary'] / 50000) * 0.5 + (data['age'] > 50) * 0.8
probs = 1 / (1 + np.exp(-logit))
data['churn'] = (np.random.rand(n_samples) < probs).astype(int)

X = data.drop('churn', axis=1)
y = data['churn']
cat_features = ['contract_type', 'payment_method']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print("Dataset sample:
", X_train.head(3))


In [ ]:
# Step 3: Train CatBoost without Manual One-Hot Encoding
try:
    cb_model = CatBoostClassifier(
        iterations=150,
        learning_rate=0.08,
        depth=5,
        cat_features=cat_features,
        verbose=False,
        random_seed=42
    )
    cb_model.fit(X_train, y_train, eval_set=(X_test, y_test))

    y_pred = cb_model.predict(X_test)
    print(f"CatBoost Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print("
Classification Report:
", classification_report(y_test, y_pred))
except NameError:
    print("Install catboost to run this cell.")


In [ ]:
# Step 4: CatBoost Feature Importance
try:
    cb_importances = pd.Series(cb_model.get_feature_importance(), index=X.columns).sort_values(ascending=False)
    plt.figure(figsize=(8, 4))
    cb_importances.plot(kind='barh', color='purple')
    plt.title("CatBoost Feature Importances")
    plt.xlabel("Importance score")
    plt.gca().invert_yaxis()
    plt.show()
except Exception as e:
    print(e)


### Summary
- **Strengths**: Best-in-class handling of categorical features, minimal parameter tuning needed out of the box, fast prediction time via oblivious trees.
- **Weaknesses**: Slower training on dense numerical data compared to LightGBM.
